In [ ]:
import json
import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def load_inverted_index(filename):
    with open(filename, 'r') as file:
        data = json.load(file)
    inverted_index = {}
    for entry in data:
        doc_ids = []
        for doc_str in entry["ids"].split(","):
            doc_str = doc_str.strip()
            if doc_str.isdigit():
                doc_ids.append(int(doc_str))
        inverted_index[entry["word"]] = doc_ids
    return inverted_index

def load_corpus(filename):
    df = pd.read_excel(filename, engine='openpyxl')
    corpus = {}
    for _, row in df.iterrows():
        doc_id = str(row["Document ID"])
        tokens = str(row["Tokens"])
        corpus[doc_id] = tokens
    return corpus

def load_queries(filename):
    queries = []
    with open(filename, 'r') as file:
        for line in file:
            entry = json.loads(line.strip())
            q_id = str(entry["_id"])
            q_text = entry["text"].lower()
            queries.append((q_id, q_text))
    return queries

def compute_tfidf_top100(query, inverted_index, corpus):

    query_tokens = query.split()
    relevant_docs = set()
    for token in query_tokens:
        if token in inverted_index:
            for docid in inverted_index[token]:
                relevant_docs.add(str(docid))

    if not relevant_docs:
        return []

    doc_texts = [corpus[doc_id] for doc_id in relevant_docs if doc_id in corpus]
    doc_ids = [doc_id for doc_id in relevant_docs if doc_id in corpus]

    vectorizer = TfidfVectorizer()
    vectors = vectorizer.fit_transform([query] + doc_texts)

    query_vector = vectors[0:1]
    doc_vectors = vectors[1:]

    sims = cosine_similarity(query_vector, doc_vectors)[0]

    scored_docs = list(zip(doc_ids, sims))
    scored_docs.sort(key=lambda x: x[1], reverse=True)

    return scored_docs[:100]

def load_neural_model():

    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    return model

def neural_rerank(query_text, top_docs, corpus, model):

    query_emb = model.encode(query_text, convert_to_numpy=True)

    reranked = []
    for (doc_id, _) in top_docs:
        doc_text = corpus.get(doc_id, "")
        doc_emb = model.encode(doc_text, convert_to_numpy=True)
        dot = np.dot(query_emb, doc_emb)
        norm_q = np.linalg.norm(query_emb)
        norm_d = np.linalg.norm(doc_emb)
        cos_sim = dot / (norm_q * norm_d + 1e-10)
        reranked.append((doc_id, cos_sim))

    reranked.sort(key=lambda x: x[1], reverse=True)
    return reranked

def main():
    inverted_index = load_inverted_index(r"scifact\inverted_index.json")
    corpus = load_corpus(r"scifact\preprocessed_corpus.xlsx")
    all_queries = load_queries(r"scifact\queries.jsonl") 
    sbert_model = load_neural_model()
    test_queries = []
    for q_id, q_text in all_queries:
        q_int = int(q_id)
        if q_int % 2 == 1 and q_int <= 1395:
            test_queries.append((q_id, q_text))
    
    print(f"Total queries loaded: {len(all_queries)}. Test queries to process: {len(test_queries)}.")

    output_file = r"scifact\assignment2_results.txt"
    run_name = "neural_run"

    with open(output_file, "w", encoding="utf-8") as outf:
        for query_id, query_text in test_queries:
            tfidf_top100 = compute_tfidf_top100(query_text, inverted_index, corpus)
            final_ranked = neural_rerank(query_text, tfidf_top100, corpus, sbert_model)
            for rank, (doc_id, score) in enumerate(final_ranked, start=1):
                outf.write(f"{query_id} Q0 {doc_id} {rank} {score:.4f} {run_name}\n")

    print(f"Done! Final results (only odd queries up to 1395) saved to {output_file}")

if __name__ == "__main__":
    main()


Total queries loaded: 1109. Test queries to process: 542.
Done! Final results (only odd queries up to 1395) saved to scifact\assignment2_results.txt


In [13]:
import pandas as pd
from sklearn.metrics import average_precision_score

def compute_map(results_file, qrels_file):

    results_df = pd.read_csv(results_file, sep=" ", header=None, names=["query_id", "Q0", "doc_id", "rank", "score", "tag"])
    qrels_df = pd.read_csv(qrels_file, sep=" ", header=None, names=["query_id", "Q0", "doc_id", "relevance"])
    query_ids = results_df["query_id"].unique()
    average_precisions = []

    for query_id in query_ids:
        retrieved_docs = results_df[results_df["query_id"] == query_id]["doc_id"].tolist()
        relevant_docs = qrels_df[qrels_df["query_id"] == query_id]["doc_id"].tolist()
        y_true = [1 if doc in relevant_docs else 0 for doc in retrieved_docs]

        if sum(y_true) > 0:
            ap = average_precision_score(y_true, list(range(len(y_true), 0, -1)))
            average_precisions.append(ap)

    map_score = sum(average_precisions) / len(average_precisions) if average_precisions else 0
    print(f"Mean Average Precision (MAP): {map_score:.4f}")

compute_map(r"scifact\assignment2_results.txt", r"scifact\qrels.txt")

Mean Average Precision (MAP): 0.5916


In [1]:
import json
import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def load_inverted_index(filename):
    with open(filename, 'r') as file:
        data = json.load(file)
    inverted_index = {}
    for entry in data:
        doc_ids = []
        for doc_str in entry["ids"].split(","):
            doc_str = doc_str.strip()
            if doc_str.isdigit():
                doc_ids.append(int(doc_str))
        inverted_index[entry["word"]] = doc_ids
    return inverted_index

def load_corpus(filename):
    df = pd.read_excel(filename, engine='openpyxl')
    corpus = {}
    for _, row in df.iterrows():
        doc_id = str(row["Document ID"])
        tokens = str(row["Tokens"])
        corpus[doc_id] = tokens
    return corpus

def load_queries(filename):
    queries = []
    with open(filename, 'r') as file:
        for line in file:
            entry = json.loads(line.strip())
            q_id = str(entry["_id"])
            q_text = entry["text"].lower()
            queries.append((q_id, q_text))
    return queries

def compute_tfidf_top100(query, inverted_index, corpus):

    query_tokens = query.split()
    relevant_docs = set()
    for token in query_tokens:
        if token in inverted_index:
            for docid in inverted_index[token]:
                relevant_docs.add(str(docid))

    if not relevant_docs:
        return []

    doc_texts = [corpus[doc_id] for doc_id in relevant_docs if doc_id in corpus]
    doc_ids = [doc_id for doc_id in relevant_docs if doc_id in corpus]

    vectorizer = TfidfVectorizer()
    vectors = vectorizer.fit_transform([query] + doc_texts)

    query_vector = vectors[0:1]
    doc_vectors = vectors[1:]

    sims = cosine_similarity(query_vector, doc_vectors)[0]

    scored_docs = list(zip(doc_ids, sims))
    scored_docs.sort(key=lambda x: x[1], reverse=True)

    return scored_docs[:100]

def load_neural_model():
    # now using all-MiniLM-L12-v2
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L12-v2')
    return model

def neural_rerank(query_text, top_docs, corpus, model):

    query_emb = model.encode(query_text, convert_to_numpy=True)

    reranked = []
    for (doc_id, _) in top_docs:
        doc_text = corpus.get(doc_id, "")
        doc_emb = model.encode(doc_text, convert_to_numpy=True)
        dot = np.dot(query_emb, doc_emb)
        norm_q = np.linalg.norm(query_emb)
        norm_d = np.linalg.norm(doc_emb)
        cos_sim = dot / (norm_q * norm_d + 1e-10)
        reranked.append((doc_id, cos_sim))

    reranked.sort(key=lambda x: x[1], reverse=True)
    return reranked

def main():
    inverted_index = load_inverted_index(r"scifact\inverted_index.json")
    corpus = load_corpus(r"scifact\preprocessed_corpus.xlsx")
    all_queries = load_queries(r"scifact\queries.jsonl") 
    sbert_model = load_neural_model()
    test_queries = []
    for q_id, q_text in all_queries:
        q_int = int(q_id)
        if q_int % 2 == 1 and q_int <= 1395:
            test_queries.append((q_id, q_text))
    
    print(f"Total queries loaded: {len(all_queries)}. Test queries to process: {len(test_queries)}.")

    output_file = r"scifact\assignment2_results_V2.txt"
    run_name = "neural_run"

    with open(output_file, "w", encoding="utf-8") as outf:
        for query_id, query_text in test_queries:
            tfidf_top100 = compute_tfidf_top100(query_text, inverted_index, corpus)
            final_ranked = neural_rerank(query_text, tfidf_top100, corpus, sbert_model)
            for rank, (doc_id, score) in enumerate(final_ranked, start=1):
                outf.write(f"{query_id} Q0 {doc_id} {rank} {score:.4f} {run_name}\n")

    print(f"Done! Final results (only odd queries up to 1395) saved to {output_file}")

if __name__ == "__main__":
    main()


c:\Users\24cya\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\24cya\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\24cya\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to r

Total queries loaded: 1109. Test queries to process: 542.
Done! Final results (only odd queries up to 1395) saved to scifact\assignment2_results_V2.txt


In [1]:
import pandas as pd
from sklearn.metrics import average_precision_score

def compute_map(results_file, qrels_file):

    results_df = pd.read_csv(results_file, sep=" ", header=None, names=["query_id", "Q0", "doc_id", "rank", "score", "tag"])
    qrels_df = pd.read_csv(qrels_file, sep=" ", header=None, names=["query_id", "Q0", "doc_id", "relevance"])
    query_ids = results_df["query_id"].unique()
    average_precisions = []

    for query_id in query_ids:
        retrieved_docs = results_df[results_df["query_id"] == query_id]["doc_id"].tolist()
        relevant_docs = qrels_df[qrels_df["query_id"] == query_id]["doc_id"].tolist()
        y_true = [1 if doc in relevant_docs else 0 for doc in retrieved_docs]

        if sum(y_true) > 0:
            ap = average_precision_score(y_true, list(range(len(y_true), 0, -1)))
            average_precisions.append(ap)

    map_score = sum(average_precisions) / len(average_precisions) if average_precisions else 0
    print(f"Mean Average Precision (MAP): {map_score:.4f}")

compute_map(r"scifact\assignment2_results_V2.txt", r"scifact\qrels.txt")

Mean Average Precision (MAP): 0.5585


# Top 10 extraction records from Q1 and Q3

In [1]:
import pandas as pd

def get_top_10_for_queries(results_file, query_ids):
    # Load results file
    results_df = pd.read_csv(results_file, sep=" ", header=None, 
                             names=["query_id", "Q0", "doc_id", "rank", "score", "run_name"])

    # Iterate over specified query IDs
    for query_id in query_ids:
        # Filter for the query and get top 10 by rank
        top_10 = results_df[results_df["query_id"] == query_id].sort_values("rank").head(10)
        
        print(f"Top 10 results for Query {query_id}:")
        print(top_10)
        print("\n")

# Example usage
results_file = "scifact/assignment2_results_V2.txt"
query_ids = [1, 3]  # Queries to analyze
get_top_10_for_queries(results_file, query_ids)


Top 10 results for Query 1:
       query_id  Q0    doc_id  rank   score    run_name
32009         1  Q0   4346436     1  0.4336  neural_run
32010         1  Q0  11172205     2  0.2945  neural_run
32011         1  Q0   3874000     3  0.2783  neural_run
32012         1  Q0  40087494     4  0.2413  neural_run
32013         1  Q0    464511     5  0.2362  neural_run
32014         1  Q0  16128711     6  0.2279  neural_run
32015         1  Q0  87430549     7  0.2272  neural_run
32016         1  Q0   8426046     8  0.2257  neural_run
32017         1  Q0  10698739     9  0.2214  neural_run
32018         1  Q0   8610932    10  0.2199  neural_run


Top 10 results for Query 3:
       query_id  Q0    doc_id  rank   score    run_name
32109         3  Q0  23389795     1  0.6056  neural_run
32110         3  Q0   4632921     2  0.4993  neural_run
32111         3  Q0   1388704     3  0.4910  neural_run
32112         3  Q0   4414547     4  0.4885  neural_run
32113         3  Q0  15570962     5  0.4754  n

# Precision Calculation

In [3]:
import pandas as pd

def compute_p_at_10(results_file, qrels_file):
    # Load files
    results_df = pd.read_csv(results_file, sep=" ", header=None, 
                             names=["query_id", "Q0", "doc_id", "rank", "score", "run_name"])
    qrels_df = pd.read_csv(qrels_file, sep=" ", header=None, 
                           names=["query_id", "Q0", "doc_id", "relevance"])

    # Filter relevance judgments (relevance > 0 is considered relevant)
    qrels_df = qrels_df[qrels_df["relevance"] > 0]

    # Get unique queries
    query_ids = results_df["query_id"].unique()

    p_at_10_scores = []

    for query_id in query_ids:
        # Get top 10 results for the query
        top_10_docs = results_df[results_df["query_id"] == query_id].sort_values("rank").head(10)["doc_id"].tolist()

        # Get relevant documents for the query
        relevant_docs = qrels_df[qrels_df["query_id"] == query_id]["doc_id"].tolist()

        # Count relevant documents in the top 10
        relevant_count = sum(1 for doc in top_10_docs if doc in relevant_docs)

        # Calculate P@10
        p_at_10 = relevant_count / 10
        p_at_10_scores.append(p_at_10)

        print(f"Query {query_id}: P@10 = {p_at_10:.4f}")

    # Average P@10 across all queries
    mean_p_at_10 = sum(p_at_10_scores) / len(p_at_10_scores)
    print(f"\nMean P@10: {mean_p_at_10:.4f}")

    return mean_p_at_10


results_file = "scifact/assignment2_results_V2.txt"
qrels_file = "scifact/qrels.txt"
compute_p_at_10(results_file, qrels_file)


Query 9: P@10 = 0.0000
Query 11: P@10 = 0.0000
Query 15: P@10 = 0.0000
Query 17: P@10 = 0.0000
Query 19: P@10 = 0.0000
Query 21: P@10 = 0.0000
Query 25: P@10 = 0.0000
Query 27: P@10 = 0.0000
Query 35: P@10 = 0.0000
Query 37: P@10 = 0.0000
Query 39: P@10 = 0.0000
Query 41: P@10 = 0.0000
Query 43: P@10 = 0.0000
Query 45: P@10 = 0.0000
Query 47: P@10 = 0.0000
Query 55: P@10 = 0.0000
Query 61: P@10 = 0.0000
Query 63: P@10 = 0.0000
Query 67: P@10 = 0.0000
Query 69: P@10 = 0.0000
Query 71: P@10 = 0.0000
Query 77: P@10 = 0.0000
Query 79: P@10 = 0.0000
Query 81: P@10 = 0.0000
Query 85: P@10 = 0.0000
Query 89: P@10 = 0.0000
Query 91: P@10 = 0.0000
Query 93: P@10 = 0.0000
Query 95: P@10 = 0.0000
Query 105: P@10 = 0.0000
Query 109: P@10 = 0.0000
Query 111: P@10 = 0.0000
Query 119: P@10 = 0.0000
Query 121: P@10 = 0.0000
Query 123: P@10 = 0.0000
Query 139: P@10 = 0.0000
Query 149: P@10 = 0.0000
Query 153: P@10 = 0.0000
Query 155: P@10 = 0.0000
Query 157: P@10 = 0.0000
Query 159: P@10 = 0.0000
Query

0.020265151515151517

# MAP Evaluation
The Mean Average Precision (MAP) evaluates the overall ranking quality. A higher MAP score indicates better alignment with the relevance judgments.
•	The all-MiniLM-L6-v2 model achieved the highest MAP score of 0.5916 compared to the all-MiniLM-L12-v2 model's 0.5585.
•	This indicates that the all-MiniLM-L6-v2 model was more effective in ranking documents relevant to the queries.
